# Chapter 4 — Neural Networks
**Track:** ML from Scratch · California Housing dataset

## Core Idea

A neural network stacks **linear → activation** blocks. Each block re-represents the data until the final prediction is (approximately) linear in the last hidden layer.

```
input x  →  [W₁·x + b₁ → ReLU]  →  [W₂·h¹ + b₂ → ReLU]  →  [W₃·h² + b₃]  →  ŷ
```

Key decisions: **activation function**, **weight initialisation**, **depth**, **width**.

## Running Example

Real estate platform — predict `MedHouseVal` (median house value, $100k units) from **all 8 features**.

| Feature | Description |
|---|---|
| MedInc | Median income (×$10k) |
| HouseAge | Median house age |
| AveRooms | Average rooms/house |
| AveBedrms | Avg bedrooms/house |
| Population | Block population |
| AveOccup | Avg occupancy |
| Latitude | Block latitude |
| Longitude | Block longitude |

Output neuron: **linear** (no activation) — house value is unbounded and continuous.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Load data
housing = fetch_california_housing()
X, y = housing.data, housing.target          # (20640, 8), (20640,)
feature_names = housing.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardise — critical for neural nets
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}")
print(f"Target range: [{y.min():.2f}, {y.max():.2f}] ($100k units)")

## Activation Functions — Math & Shapes

| Activation | Formula | Output range | Best for |
|---|---|---|---|
| ReLU | $\max(0, z)$ | $[0, \infty)$ | hidden layers (default) |
| Sigmoid | $\frac{1}{1+e^{-z}}$ | $(0,1)$ | binary output |
| Tanh | $\frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1,1)$ | hidden (zero-centred) |
| Softmax | $\frac{e^{z_k}}{\sum_j e^{z_j}}$ | $(0,1)$, sums to 1 | multi-class output |
| Linear | $z$ | $(-\infty, \infty)$ | regression output |

In [ ]:
# Plot all activation functions side by side
z = np.linspace(-4, 4, 300)

activations = {
    'ReLU':    np.maximum(0, z),
    'Sigmoid': 1 / (1 + np.exp(-z)),
    'Tanh':    np.tanh(z),
    'Linear':  z,
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, vals) in zip(axes, activations.items()):
    ax.plot(z, vals, linewidth=2)
    ax.axhline(0, color='grey', linewidth=0.5)
    ax.axvline(0, color='grey', linewidth=0.5)
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('z (pre-activation)')
    ax.set_xlim(-4, 4)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('g(z)')
plt.suptitle('Activation Functions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Weight Initialisation — Xavier vs He

**Xavier / Glorot** (for Sigmoid / Tanh):
$$W \sim \mathcal{U}\!\left(-\sqrt{\frac{6}{n_\text{in}+n_\text{out}}},\ \sqrt{\frac{6}{n_\text{in}+n_\text{out}}}\right)$$

**He** (for ReLU):
$$W \sim \mathcal{N}\!\left(0,\ \sqrt{\frac{2}{n_\text{in}}}\right)$$

Both are calibrated so the **variance of activations stays roughly constant** across layers — preventing signal from exploding or vanishing before any gradient update.

In [ ]:
# Visualise how initialisation strategy affects activation variance across layers
rng = np.random.default_rng(42)
x_demo = rng.normal(0, 1, (1000, 8))   # 1000 fake samples, 8 features
layer_sizes = [8, 64, 64, 64, 64, 1]

def forward_variance(x, layer_sizes, init='he', activation=np.tanh):
    """Track per-layer activation std dev under a given init strategy."""
    stds = []
    h = x
    for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        if init == 'he':
            W = rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out))
        elif init == 'xavier':
            limit = np.sqrt(6 / (n_in + n_out))
            W = rng.uniform(-limit, limit, (n_in, n_out))
        elif init == 'zeros':
            W = np.zeros((n_in, n_out))
        else:  # large random
            W = rng.normal(0, 2.0, (n_in, n_out))
        b = np.zeros(n_out)
        h = activation(h @ W + b)
        stds.append(h.std())
    return stds

layer_labels = ['Layer 1', 'Layer 2', 'Layer 3', 'Layer 4']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
scenarios = [
    ('zeros',  'Zero init (symmetry problem)', 'red',   np.maximum(0, np.array([0.0]* 4))),
    ('xavier', 'Xavier init (Tanh layers)',   'blue',  None),
    ('he',     'He init (ReLU layers)',        'green', None),
]

for ax, (init, title, color, override) in zip(axes, scenarios):
    act = np.tanh if init == 'xavier' else np.maximum
    if init == 'xavier':
        stds = forward_variance(x_demo, layer_sizes, init='xavier', activation=np.tanh)
    elif init == 'he':
        stds = forward_variance(x_demo, layer_sizes, init='he',
                                activation=lambda z: np.maximum(0, z))
    else:  # zeros — all activations collapse to same value
        stds = [0.0, 0.0, 0.0, 0.0]

    ax.bar(layer_labels, stds[:4], color=color, alpha=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Activation std dev')
    ax.set_ylim(0, 1.2)
    ax.axhline(0, color='grey', linewidth=0.5)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Weight Initialisation: Effect on Activation Variance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Xavier stds:", [f"{s:.3f}" for s in forward_variance(x_demo, layer_sizes,'xavier',np.tanh)[:4]])
print("He stds:    ", [f"{s:.3f}" for s in forward_variance(x_demo, layer_sizes,'he', lambda z: np.maximum(0,z))[:4]])

## Manual Forward Pass (NumPy)

Build the full forward pass from scratch with He-initialised weights and ReLU hidden layers.
Output layer: **linear** (regression).

In [ ]:
def relu(z):
    return np.maximum(0, z)

def he_init(n_in, n_out, rng):
    """He initialisation: N(0, sqrt(2/n_in)). Designed for ReLU layers."""
    return rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out))

rng = np.random.default_rng(42)

# Architecture: 8 → 128 → 64 → 1
W1 = he_init(8, 128, rng);   b1 = np.zeros(128)
W2 = he_init(128, 64, rng);  b2 = np.zeros(64)
W3 = he_init(64, 1, rng);    b3 = np.zeros(1)

def forward(X):
    """Forward pass: two ReLU hidden layers, linear output.
    X: (n_samples, 8) — must be standardised first.
    Returns: (n_samples,) predictions.
    """
    h1 = relu(X @ W1 + b1)           # (n, 128)
    h2 = relu(h1 @ W2 + b2)          # (n, 64)
    return (h2 @ W3 + b3).ravel()    # (n,)  — no activation on output

y_hat_init = forward(X_test_s)
print("Predictions (random init, first 5):", y_hat_init[:5].round(3))
print("Actual values (first 5):           ", y_test[:5].round(3))
print(f"R² before training: {r2_score(y_test, y_hat_init):.4f}  (random, expect near 0 or negative)")

## Baseline Comparison — Linear Regression vs Neural Net

Before tuning the network, establish how much linear regression gets us.

In [ ]:
# Baseline: Linear Regression (Ch.1)
lr = LinearRegression().fit(X_train_s, y_train)
r2_lr = r2_score(y_test, lr.predict(X_test_s))
rmse_lr = mean_squared_error(y_test, lr.predict(X_test_s), squared=False)

# Neural Network: sklearn MLPRegressor (2 hidden layers, 128 + 64)
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
)
mlp.fit(X_train_s, y_train)
r2_mlp = r2_score(y_test, mlp.predict(X_test_s))
rmse_mlp = mean_squared_error(y_test, mlp.predict(X_test_s), squared=False)

print(f"{'Model':<25} {'R²':>8} {'RMSE':>10}")
print("-" * 45)
print(f"{'Linear Regression':<25} {r2_lr:>8.4f} {rmse_lr:>10.4f}")
print(f"{'MLP (128→64→1)':<25} {r2_mlp:>8.4f} {rmse_mlp:>10.4f}")
print(f"\nGain: ΔR² = {r2_mlp - r2_lr:+.4f}")

In [ ]:
# Training loss curve (MLPRegressor stores loss_curve_)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve
axes[0].plot(mlp.loss_curve_, label='Training loss', color='steelblue')
if hasattr(mlp, 'validation_scores_'):
    # validation_scores_ is R², negate to make it a loss proxy for comparison
    axes[0].plot(
        [-s for s in mlp.validation_scores_],
        label='Validation loss (−R²)', color='coral', linestyle='--'
    )
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Predicted vs Actual scatter
y_pred_mlp = mlp.predict(X_test_s)
axes[1].scatter(y_test, y_pred_mlp, alpha=0.3, s=10, color='steelblue')
mn, mx = y_test.min(), y_test.max()
axes[1].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual MedHouseVal ($100k)')
axes[1].set_ylabel('Predicted MedHouseVal ($100k)')
axes[1].set_title(f'Predicted vs Actual (R²={r2_mlp:.3f})')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Depth vs Width Sweep

How do different architectures compare? We test width (units per layer, assuming 2 equal layers) and depth (number of layers at fixed width 64).

In [ ]:
# Width sweep: 2 hidden layers, varying units
widths = [8, 16, 32, 64, 128, 256]
width_r2 = []
for w in widths:
    m = MLPRegressor(
        hidden_layer_sizes=(w, w), activation='relu',
        solver='adam', max_iter=300, random_state=42
    ).fit(X_train_s, y_train)
    width_r2.append(r2_score(y_test, m.predict(X_test_s)))

# Depth sweep: 1–5 hidden layers of 64 units each
depths = [1, 2, 3, 4, 5]
depth_r2 = []
for d in depths:
    m = MLPRegressor(
        hidden_layer_sizes=tuple([64] * d), activation='relu',
        solver='adam', max_iter=300, random_state=42
    ).fit(X_train_s, y_train)
    depth_r2.append(r2_score(y_test, m.predict(X_test_s)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(widths, width_r2, marker='o', color='steelblue')
axes[0].set_xscale('log', base=2)
axes[0].set_xticks(widths)
axes[0].set_xticklabels(widths)
axes[0].set_xlabel('Units per layer (2 hidden layers)')
axes[0].set_ylabel('R² (test)')
axes[0].set_title('Width Sweep')
axes[0].grid(alpha=0.3)

axes[1].plot(depths, depth_r2, marker='s', color='coral')
axes[1].set_xticks(depths)
axes[1].set_xlabel('Number of hidden layers (64 units each)')
axes[1].set_ylabel('R² (test)')
axes[1].set_title('Depth Sweep')
axes[1].grid(alpha=0.3)

plt.suptitle('Depth vs Width: R² on California Housing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Width R² values:", [f"{r:.3f}" for r in width_r2])
print("Depth R² values:", [f"{r:.3f}" for r in depth_r2])

## Trap 1 — Wrong Output Activation (Regression)

Using **sigmoid** on the output for regression caps all predictions in (0, 1). House values range up to 5.0 ($500k) — sigmoid will systematically clip everything above ~1.

**Correct choice:** linear output (no activation).

In [ ]:
# Simulate the sigmoid-output trap with a manual forward pass
def forward_with_sigmoid_output(X):
    """Intentionally wrong: sigmoid on regression output."""
    h1 = relu(X @ W1 + b1)
    h2 = relu(h1 @ W2 + b2)
    z_out = (h2 @ W3 + b3).ravel()
    return 1 / (1 + np.exp(-z_out))   # sigmoid squashes to (0,1)

y_sigmoid = forward_with_sigmoid_output(X_test_s)
y_linear  = forward(X_test_s)         # correct: no output activation

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (preds, label, color) in zip(axes, [
    (y_sigmoid, 'Sigmoid output (WRONG for regression)', 'red'),
    (y_linear,  'Linear output (correct for regression)', 'steelblue'),
]):
    ax.scatter(y_test, preds, alpha=0.3, s=10, color=color)
    mn, mx = y_test.min(), y_test.max()
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1)
    ax.set_xlabel('Actual MedHouseVal')
    ax.set_ylabel('Predicted MedHouseVal')
    ax.set_title(label)
    ax.set_ylim(-0.5, 6)
    ax.grid(alpha=0.3)

plt.suptitle('Output Activation Trap: Sigmoid vs Linear', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Prediction range (sigmoid): [{y_sigmoid.min():.3f}, {y_sigmoid.max():.3f}]  <- clipped")
print(f"Prediction range (linear):  [{y_linear.min():.3f},  {y_linear.max():.3f}]")

## Trap 2 — Zero Initialisation (Symmetry)

If all weights are zero, every neuron in a layer receives the **same gradient** and updates identically — the entire layer effectively has width 1 regardless of how many units you declare.

In [ ]:
# Demo: zero-init vs He-init — check that all neurons in layer 1 are identical
W1_zero = np.zeros((8, 128))
b1_zero = np.zeros(128)

# Forward one sample
x_single = X_test_s[:1]   # shape (1, 8)

h1_zero = relu(x_single @ W1_zero + b1_zero)   # (1, 128)
h1_he   = relu(x_single @ W1 + b1)              # (1, 128)

print("Zero-init — are all 128 neurons identical?")
print(f"  All same value: {np.allclose(h1_zero, h1_zero[0, 0])}")
print(f"  Unique activations: {len(np.unique(h1_zero.round(8)))}")

print("\nHe-init — are all 128 neurons identical?")
print(f"  All same value: {np.allclose(h1_he, h1_he[0, 0])}")
print(f"  Unique activations: {len(np.unique(h1_he.round(8)))}")

# Visualise the neuron activation distributions
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].bar(range(20), h1_zero[0, :20], color='red', alpha=0.7)
axes[0].set_title('Zero init — first 20 neuron activations (all identical)')
axes[0].set_xlabel('Neuron index'); axes[0].set_ylabel('Activation')
axes[1].bar(range(20), h1_he[0, :20], color='steelblue', alpha=0.7)
axes[1].set_title('He init — first 20 neuron activations (diverse)')
axes[1].set_xlabel('Neuron index'); axes[1].set_ylabel('Activation')
plt.tight_layout()
plt.show()

## Exercises

**Exercise 1 — Activation ablation**  
Replace the hidden activation from `relu` to `tanh` in `MLPRegressor`. Does R² improve or decline on housing?

**Exercise 2 — Feature importance via weight magnitude**  
After fitting the MLP, the first-layer weight matrix `mlp.coefs_[0]` has shape (8, 128). Compute the mean absolute weight per input feature and plot a bar chart. Which feature is most predictive?

**Exercise 3 — Unscaled input trap**  
Fit an MLPRegressor **without** calling `StandardScaler` first. Compare R² to the scaled version and plot residuals. What changes?

In [ ]:
# Exercise 1 scaffold — activation ablation
# Replace 'relu' with 'tanh' and compare R²

mlp_tanh = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',      # <-- change to 'tanh'
    solver='adam',
    max_iter=300,
    random_state=42,
)
mlp_tanh.fit(X_train_s, y_train)
r2_tanh = r2_score(y_test, mlp_tanh.predict(X_test_s))
print(f"R² (relu): {r2_mlp:.4f}")
print(f"R² (tanh): {r2_tanh:.4f}")
# TODO: change activation and observe difference

In [ ]:
# Exercise 2 scaffold — feature importance via first-layer weight magnitude
# mlp.coefs_[0] has shape (n_input_features, n_hidden_units_layer_1)

W_first = mlp.coefs_[0]                         # (8, 128)
importance = np.abs(W_first).mean(axis=1)        # mean absolute weight per input feature

# TODO: plot a horizontal bar chart of importance vs feature_names
# Hint: plt.barh(feature_names, importance[sorted_idx])

In [ ]:
# Exercise 3 scaffold — unscaled input trap
# Fit MLPRegressor on raw (un-standardised) X_train and measure R²

mlp_unscaled = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42,
)
mlp_unscaled.fit(X_train, y_train)   # raw features, no StandardScaler
r2_unscaled = r2_score(y_test, mlp_unscaled.predict(X_test))
print(f"R² (scaled):   {r2_mlp:.4f}")
print(f"R² (unscaled): {r2_unscaled:.4f}")
# TODO: plot residuals for both and compare distributions